In [1]:
import os

print("INPUT ROOT:")
print(os.listdir("/kaggle/input"))

print("\nDATASETS:")
print(os.listdir("/kaggle/input/datasets"))

print("\nSAUTKIN DATASETS:")
print(os.listdir("/kaggle/input/datasets/sautkin"))

INPUT ROOT:
['datasets']

DATASETS:
['sautkin']

SAUTKIN DATASETS:
['imagenet1kvalid', 'imagenet1k2', 'imagenet1k0', 'imagenet1k3', 'imagenet1k1']


## Clean the Working Directory

In [2]:
import shutil
import os

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

In [3]:
!rm -rf LSNET-advanced
!git clone -b pretrained https://github.com/Param45/LSNET-advanced.git
%cd LSNET-advanced
!ls

Cloning into 'LSNET-advanced'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 254 (delta 12), reused 14 (delta 6), pack-reused 228 (from 1)
Receiving objects: 100% (254/254), 41.51 MiB | 39.84 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/kaggle/working/LSNET-advanced
data		figures		  logs	     README.md		   segmentation
detection	flops.py	  losses.py  README_robustness.md  speed.py
engine.py	kaggle_config.py  main.py    requirements.txt	   train.sh
eval_robust.sh	kaggle_run.py	  model      robust.py		   utils.py
eval.sh		KAGGLE_SETUP.md   pretrain   robust_utils.py


In [4]:
import torch
print(torch.__version__)

2.10.0+cu128


In [5]:
!pip install -q timm fvcore wandb
!pip install timm==0.5.4 einops==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but yo

In [6]:
from kaggle_config import IMAGENET_PATH

print(IMAGENET_PATH)

/kaggle/input/datasets/sautkin


In [7]:
import os

for ds in [
    "imagenet1k0",
    "imagenet1k1",
    "imagenet1k2",
    "imagenet1k3",
    "imagenet1kvalid"
]:
    path = f"/kaggle/input/datasets/sautkin/{ds}"
    print(ds, len(os.listdir(path)))

imagenet1k0 500
imagenet1k1 500
imagenet1k2 500
imagenet1k3 500
imagenet1kvalid 1000


In [8]:
!python main.py --help

usage: main.py [-h] [--batch-size BATCH_SIZE] [--epochs EPOCHS]
               [--model MODEL] [--input-size INPUT_SIZE] [--sparse-ska]
               [--sparse-top-k SPARSE_TOP_K] [--model-ema] [--no-model-ema]
               [--model-ema-decay MODEL_EMA_DECAY]
               [--model-ema-steps MODEL_EMA_STEPS] [--model-ema-force-cpu]
               [--opt OPTIMIZER] [--opt-eps EPSILON]
               [--opt-betas BETA [BETA ...]] [--clip-grad NORM]
               [--clip-mode CLIP_MODE] [--momentum M]
               [--weight-decay WEIGHT_DECAY] [--sched SCHEDULER] [--lr LR]
               [--lr-noise pct, pct [pct, pct ...]] [--lr-noise-pct PERCENT]
               [--lr-noise-std STDDEV] [--warmup-lr LR] [--min-lr LR]
               [--decay-epochs N] [--warmup-epochs N] [--cooldown-epochs N]
               [--patience-epochs N] [--decay-rate RATE] [--ThreeAugment]
               [--color-jitter PCT] [--aa NAME] [--smoothing SMOOTHING]
               [--train-interpolation TRAIN_INT

In [9]:
from data.datasets import build_dataset

In [10]:
from types import SimpleNamespace
import time

args = SimpleNamespace(
    data_set="IMNET",
    data_path="/kaggle/input/datasets/sautkin",
    input_size=224,
    color_jitter=0.4,
    aa='rand-m9-mstd0.5-inc1',
    train_interpolation='bicubic',
    reprob=0.25,
    remode='pixel',
    recount=1,
    finetune='',
    inat_category='name',
    dataset_fraction=0.4
)

start = time.time()

dataset_train, n_classes = build_dataset(
    is_train=True,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_train))
print("Time:", time.time() - start)

Classes: 1000
Samples: 511822
Time: 38.3290696144104


In [11]:
start = time.time()

dataset_val, n_classes = build_dataset(
    is_train=False,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_val))
print("Time:", time.time() - start)

Classes: 1000
Samples: 20000
Time: 9.737542629241943


In [12]:
!python kaggle_run.py --help

usage: kaggle_run.py [-h] --action
                     {train_clf,eval_clf,train_det,test_det,train_seg,test_seg,robust_clf}
                     [--model MODEL] [--config CONFIG]
                     [--checkpoint CHECKPOINT] [--resume RESUME]
                     [--extra-args EXTRA_ARGS]

Kaggle execution entrypoint helper

options:
  -h, --help            show this help message and exit
  --action {train_clf,eval_clf,train_det,test_det,train_seg,test_seg,robust_clf}
                        Action to perform
  --model MODEL         Model name
  --config CONFIG       Path to MMCV/MMDet/MMSeg config file
  --checkpoint CHECKPOINT
                        Path to checkpoint file
  --resume RESUME       Path to checkpoint to resume training from
  --extra-args EXTRA_ARGS
                        Extra arguments to pass to the script


In [13]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [14]:
# To train from scratch

# !NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 torchrun --nproc_per_node=2 main.py --model lsnet_t --data-set IMNET

In [15]:
# To fake fine-tune for 5 epochs for Proposal 3 with 100% dataset

# %cd /kaggle/working/LSNET-advanced
# !python kaggle_run.py --action train_clf --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 5 --warmup-epochs 1 --sparse-ska --sparse-top-k 5 --lr 5e-5


In [16]:
# To fake fine-tune for 5 epochs for Proposal 3 with 40% dataset

%cd /kaggle/working/LSNET-advanced
!python kaggle_run.py --action train_clf --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 5 --warmup-epochs 2 --lr 5e-5 --sparse-ska --sparse-top-k 5 --dataset-fraction 0.4


/kaggle/working/LSNET-advanced
Executing: torchrun --nproc_per_node=2 main.py --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 5 --warmup-epochs 2 --lr 5e-5 --sparse-ska --sparse-top-k 5 --dataset-fraction 0.4
W0616 11:44:48.293000 182 torch/distributed/run.py:852]
W0616 11:44:48.293000 182 torch/distributed/run.py:852] *****************************************
W0616 11:44:48.293000 182 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
W0616 11:44:48.293000 182 torch/distributed/run.py:852] *****************************************
| distributed init (rank 0): env://
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mu